In [0]:
df = spark.range(1000000)
df.filter(df.id % 2 == 0).count()


# ============================================================
# Day 11 - Spark UI (adapted for Databricks Free Edition)
# ============================================================
# Free Edition runs on SERVERLESS compute, which does not have the
# classic Jobs / Stages / Storage tabs from traditional Databricks.
# Those tabs only exist on Assigned/Classic clusters.
#
# Serverless equivalent found today:
#   - Query History  -> equivalent of the Jobs tab (one row per action)
#   - Query plan diagram (via "View all in query history" -> expand a query)
#                     -> equivalent of the Stages tab (shows task/partition count)
#   - "Memory peak" toggle on the query plan
#                     -> equivalent of the Storage tab (shows memory per step,
#                        not a persistent cached-table view)
# ============================================================

df = spark.range(1000000)
df.filter(df.id % 2 == 0).count()
# Confirmed via query plan: Range (1M) -> Filter (500K, exactly half - correct)
# -> Aggregate (8 partial counts, one per partition) -> Shuffle -> Aggregate (1 final count)

df2 = df.repartition(8)
df2.filter(df2.id % 2 == 0).count()
# The "8" appearing at the Aggregate/Shuffle steps in the query plan IS the
# partition count from Day 9's repartition(8) - now seen directly, not just described.

# Memory peak view showed Shuffle using ~1.03 GB, while every other step showed ~0 bytes.
# This matches Day 9's warning that repartition()/shuffles are the genuinely
# expensive operation - actual data movement between partitions, now visible in bytes.

df3 = spark.range(500000)
df3.cache()
df3.count()
# Note: Free Edition serverless doesn't expose a persistent Storage tab to confirm
# this cache visually the way a Classic cluster would. Conceptually still valid:
# cache() stores the result after the first action; a second action would reuse it.

# One sentence for the plan's practice task:
# If a colleague's PySpark job was running slowly on a Classic cluster, I'd check
# the Stages tab for one task taking far longer than the others, then the Storage
# tab to confirm caching is behaving as expected. On Free Edition serverless, the
# equivalent check is the query plan diagram (task/partition counts) and the
# Memory peak toggle (per-step cost, e.g. an expensive Shuffle).